## Ingestão e Exploração da Camada Bronze

Este notebook contempla a obtenção dos dados, sua ingestão na camada Bronze, o profiling utilizado para catalogação das fontes e as análises exploratórias que subsidiam as decisões de modelagem e qualidade de dados do projeto.

### 01_carga_dados
Nesta etapa é realizada a obtenção do dataset **Brazilian E-Commerce Public Dataset by Olist** diretamento do Kaggle utilizandoa biblioteza `kagglehub`.

In [0]:
# Inicialmente, a biblioteca `kagglehub` foi instalada no ambiente Databricks:
%pip install kagglehub

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
#Dataset foi obtido diretamente do Kaggle. O comando realiza o download da versão disponível do dataset e retorna o diretório em que os arquivos foram disponibilizados no ambiente Databricks.

import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

100%|██████████| 42.6M/42.6M [00:00<00:00, 65.1MB/s]

Extracting files...


Path to dataset files: /home/spark-2b97847a-4cfd-4147-9d81-f3/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2


In [0]:
# Verificação dos arquivos disponibilizados

import os
arquivos = os.listdir(path)
for arquivos in arquivos:
  print(arquivos)

olist_sellers_dataset.csv
olist_order_items_dataset.csv
olist_customers_dataset.csv
olist_order_reviews_dataset.csv
product_category_name_translation.csv
olist_geolocation_dataset.csv
olist_order_payments_dataset.csv
olist_products_dataset.csv
olist_orders_dataset.csv


### 02_ingestao_bronze
Nesta etapa os arquivos selecionados são carregados no Databricks e persistidos como tabelas Delta na camada **Bronze**. Essa camada preserva a estrutura e a granularidade dos dados de origem, servindo como ponto de partida para as etapas posteriores de exploração.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze

In [0]:
import os
import pandas as pd

arquivos_bronze = {
    "olist_orders_dataset.csv": "bronze.orders",
    "olist_order_items_dataset.csv": "bronze.order_items",
    "olist_products_dataset.csv": "bronze.products",
    "olist_customers_dataset.csv": "bronze.customers",
    "olist_sellers_dataset.csv": "bronze.sellers",
    "olist_order_reviews_dataset.csv": "bronze.order_reviews",
    "olist_order_payments_dataset.csv": "bronze.order_payments",
    "product_category_name_translation.csv": "bronze.category_translation"
}

for arquivo, tabela in arquivos_bronze.items():

    caminho_arquivo = os.path.join(path, arquivo)

    # Leitura do CSV no armazenamento local
    df_pandas = pd.read_csv(caminho_arquivo)

    # Conversão para DataFrame Spark
    df_spark = spark.createDataFrame(df_pandas)

    # Persistência como tabela Delta na camada Bronze
    (
        df_spark.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabela)
    )

    print(f"{arquivo} → {tabela}")

olist_orders_dataset.csv → bronze.orders
olist_order_items_dataset.csv → bronze.order_items
olist_products_dataset.csv → bronze.products
olist_customers_dataset.csv → bronze.customers
olist_sellers_dataset.csv → bronze.sellers
olist_order_reviews_dataset.csv → bronze.order_reviews
olist_order_payments_dataset.csv → bronze.order_payments
product_category_name_translation.csv → bronze.category_translation


### 03_catalogo_dados
Nesta etapa é realizado o 'reconhecimento' das tabelas da camada Bronze para apoiar sua catalogação. São avaliados aspectos como volume de registros, unicidade das chaves, valores nulos, intervalos de valores e domínios de campos categóricos. A partir dessas informações, as tabelas e seus campos são documentados no Unity Catalog.

In [0]:
%sql
-- query para trazer os tipos de dados de cada campo da tabela, no caso o formato que os dados vieram da base bruta da Olist
SELECT
    table_name,
    column_name,
    data_type,
    ordinal_position
FROM information_schema.columns
WHERE table_schema = 'bronze'
ORDER BY table_name, ordinal_position;

table_name,column_name,data_type,ordinal_position
category_translation,product_category_name,STRING,0
category_translation,product_category_name_english,STRING,1
customers,customer_id,STRING,0
customers,customer_unique_id,STRING,1
customers,customer_zip_code_prefix,LONG,2
customers,customer_city,STRING,3
customers,customer_state,STRING,4
order_items,order_id,STRING,0
order_items,order_item_id,LONG,1
order_items,product_id,STRING,2


#### 03_1_orders

In [0]:
%sql
-- query pra levantar os dominios da base orders
SELECT
    -- Domínio de status
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT order_id) AS qtd_order_id,
    COUNT(DISTINCT customer_id) AS qtd_customer_id,

    -- Intervalos de datas
    MIN(order_purchase_timestamp) AS min_purchase,
    MAX(order_purchase_timestamp) AS max_purchase,

    MIN(order_approved_at) AS min_approved,
    MAX(order_approved_at) AS max_approved,

    MIN(order_delivered_carrier_date) AS min_carrier,
    MAX(order_delivered_carrier_date) AS max_carrier,

    MIN(order_delivered_customer_date) AS min_delivery,
    MAX(order_delivered_customer_date) AS max_delivery,

    MIN(order_estimated_delivery_date) AS min_estimated,
    MAX(order_estimated_delivery_date) AS max_estimated,

    -- Nulos
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
    SUM(CASE WHEN order_status IS NULL THEN 1 ELSE 0 END) AS null_status,
    SUM(CASE WHEN order_approved_at IS NULL THEN 1 ELSE 0 END) AS null_approved,
    SUM(CASE WHEN order_delivered_carrier_date IS NULL THEN 1 ELSE 0 END) AS null_carrier,
    SUM(CASE WHEN order_delivered_customer_date IS NULL THEN 1 ELSE 0 END) AS null_delivery
FROM bronze.orders;

qtd_registros,qtd_order_id,qtd_customer_id,min_purchase,max_purchase,min_approved,max_approved,min_carrier,max_carrier,min_delivery,max_delivery,min_estimated,max_estimated,null_order_id,null_customer_id,null_status,null_approved,null_carrier,null_delivery
99441,99441,99441,2016-09-04 21:15:19,2018-10-17 17:30:18,2016-09-15 12:16:38,2018-09-03 17:40:06,2016-10-08 10:34:01,2018-09-11 19:48:28,2016-10-11 13:46:32,2018-10-17 13:22:46,2016-09-30 00:00:00,2018-11-12 00:00:00,0,0,0,160,1783,2965


In [0]:
%sql
-- query para levantar dados categorios da base orders
SELECT
    order_status,
    COUNT(*) AS qtd_pedidos
FROM bronze.orders
GROUP BY order_status
ORDER BY qtd_pedidos DESC;

order_status,qtd_pedidos
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [0]:
%sql
-- Incluindo descrição da base orders no meu Catalog
COMMENT ON TABLE bronze.orders IS
'Dados brutos dos pedidos do dataset Brazilian E-Commerce Public Dataset by Olist. Possui uma linha por pedido e contém identificadores, status e datas relacionadas ao ciclo do pedido.';

In [0]:
%sql
-- Incluindo descrição dos campos, com dominio e identificadores de nullos da base order no meu Catalog

ALTER TABLE bronze.orders ALTER COLUMN order_id
COMMENT 'Identificador único do pedido. Domínio: identificador alfanumérico único e não nulo.';

ALTER TABLE bronze.orders ALTER COLUMN customer_id
COMMENT 'Identificador do cliente associado ao pedido. Domínio: identificador alfanumérico não nulo.';

ALTER TABLE bronze.orders ALTER COLUMN order_status
COMMENT 'Status do pedido. Domínio observado: delivered, shipped, canceled, unavailable, invoiced, processing, created e approved. Não apresenta valores nulos.';

ALTER TABLE bronze.orders ALTER COLUMN order_purchase_timestamp
COMMENT 'Data e hora em que o pedido foi realizado. Domínio observado: 2016-09-04 21:15:19 a 2018-10-17 17:30:18. Não apresenta valores nulos. Na camada Bronze é mantido como STRING.';

ALTER TABLE bronze.orders ALTER COLUMN order_approved_at
COMMENT 'Data e hora de aprovação do pedido. Domínio observado: 2016-09-15 12:16:38 a 2018-09-03 17:40:06. Pode apresentar valores nulos. Na camada Bronze é mantido como STRING.';

ALTER TABLE bronze.orders ALTER COLUMN order_delivered_carrier_date
COMMENT 'Data e hora em que o pedido foi encaminhado à transportadora. Domínio observado: 2016-10-08 10:34:01 a 2018-09-11 19:48:28. Pode apresentar valores nulos. Na camada Bronze é mantido como STRING.';

ALTER TABLE bronze.orders ALTER COLUMN order_delivered_customer_date
COMMENT 'Data e hora de entrega do pedido ao cliente. Domínio observado: 2016-10-11 13:46:32 a 2018-10-17 13:22:46. Pode apresentar valores nulos. Na camada Bronze é mantido como STRING.';

ALTER TABLE bronze.orders ALTER COLUMN order_estimated_delivery_date
COMMENT 'Data estimada para entrega do pedido ao cliente. Domínio observado: 2016-09-30 00:00:00 a 2018-11-12 00:00:00. Não apresenta valores nulos. Na camada Bronze é mantido como STRING.';

#### 03_2_orders_itens

In [0]:
%sql
-- query pra levantar os dominios da base orders_itens


SELECT
    COUNT(*) AS qtd_registros,

    -- Identificadores
    COUNT(DISTINCT order_id) AS qtd_order_id,
    COUNT(DISTINCT product_id) AS qtd_product_id,
    COUNT(DISTINCT seller_id) AS qtd_seller_id,

    -- Domínio order_item_id
    MIN(order_item_id) AS min_order_item_id,
    MAX(order_item_id) AS max_order_item_id,

    -- Domínio da data
    MIN(shipping_limit_date) AS min_shipping_limit_date,
    MAX(shipping_limit_date) AS max_shipping_limit_date,

    -- Domínio dos valores
    MIN(price) AS min_price,
    MAX(price) AS max_price,

    MIN(freight_value) AS min_freight_value,
    MAX(freight_value) AS max_freight_value,

    -- Nulos
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN order_item_id IS NULL THEN 1 ELSE 0 END) AS null_order_item_id,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
    SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END) AS null_seller_id,
    SUM(CASE WHEN shipping_limit_date IS NULL THEN 1 ELSE 0 END) AS null_shipping_limit_date,
    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS null_price,
    SUM(CASE WHEN freight_value IS NULL THEN 1 ELSE 0 END) AS null_freight_value

FROM bronze.order_items;

qtd_registros,qtd_order_id,qtd_product_id,qtd_seller_id,min_order_item_id,max_order_item_id,min_shipping_limit_date,max_shipping_limit_date,min_price,max_price,min_freight_value,max_freight_value,null_order_id,null_order_item_id,null_product_id,null_seller_id,null_shipping_limit_date,null_price,null_freight_value
112650,98666,32951,3095,1,21,2016-09-19 00:15:34,2020-04-09 22:35:08,0.85,6735.0,0.0,409.68,0,0,0,0,0,0,0


In [0]:
%sql
-- query para confirmar granularidade da base
SELECT
    COUNT(*) AS qtd_registros,
    COUNT(
        DISTINCT CONCAT(
            order_id,
            '-',
            CAST(order_item_id AS STRING)
        )
    ) AS qtd_chaves_order_item,
    COUNT(*) -
    COUNT(
        DISTINCT CONCAT(
            order_id,
            '-',
            CAST(order_item_id AS STRING)
        )
    ) AS qtd_duplicidades
FROM bronze.order_items;

qtd_registros,qtd_chaves_order_item,qtd_duplicidades
112650,112650,0


In [0]:
%sql
-- Incluindo descrição da base order_items no meu Catalog

COMMENT ON TABLE bronze.order_items IS
'Dados brutos dos itens associados aos pedidos do dataset Brazilian E-Commerce Public Dataset by Olist. Possui uma linha por item dentro de cada pedido, sendo a combinação order_id e order_item_id única na tabela. Origem: olist_order_items_dataset.csv.';

In [0]:
%sql

-- Incluindo descrição dos campos, com dominio e identificadores de nullos da base order_items no meu Catalog

ALTER TABLE bronze.order_items ALTER COLUMN order_id
COMMENT 'Identificador do pedido ao qual o item pertence. Domínio: identificador alfanumérico não nulo. Foram observados 98.666 pedidos distintos.';

ALTER TABLE bronze.order_items ALTER COLUMN order_item_id
COMMENT 'Identificador sequencial do item dentro do pedido. Domínio observado: valores inteiros de 1 a 21, sem valores nulos. Em conjunto com order_id identifica unicamente um item do pedido.';

ALTER TABLE bronze.order_items ALTER COLUMN product_id
COMMENT 'Identificador do produto associado ao item do pedido. Domínio: identificador alfanumérico não nulo. Foram observados 32.951 produtos distintos.';

ALTER TABLE bronze.order_items ALTER COLUMN seller_id
COMMENT 'Identificador do vendedor responsável pelo item. Domínio: identificador alfanumérico não nulo. Foram observados 3.095 vendedores distintos.';

ALTER TABLE bronze.order_items ALTER COLUMN shipping_limit_date
COMMENT 'Data e hora limite para envio do item pelo vendedor. Domínio observado: 2016-09-19 00:15:34 a 2020-04-09 22:35:08, sem valores nulos. Na camada Bronze é mantido como STRING.';

ALTER TABLE bronze.order_items ALTER COLUMN price
COMMENT 'Preço do item, sem inclusão do valor de frete. Domínio observado: 0,85 a 6.735,00, sem valores nulos.';

ALTER TABLE bronze.order_items ALTER COLUMN freight_value
COMMENT 'Valor de frete associado ao item do pedido. Domínio observado: 0,00 a 409,68, sem valores nulos.';

#### 03_3_products

In [0]:
%sql
-- query pra levantar os dominios da base products

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT product_id) AS qtd_product_id,
    COUNT(DISTINCT product_category_name) AS qtd_categorias,

    -- Comprimento dos textos
    MIN(product_name_lenght) AS min_name_lenght,
    MAX(product_name_lenght) AS max_name_lenght,

    MIN(product_description_lenght) AS min_description_lenght,
    MAX(product_description_lenght) AS max_description_lenght,

    -- Fotos
    MIN(product_photos_qty) AS min_photos_qty,
    MAX(product_photos_qty) AS max_photos_qty,

    -- Peso e dimensões
    MIN(product_weight_g) AS min_weight_g,
    MAX(product_weight_g) AS max_weight_g,

    MIN(product_length_cm) AS min_length_cm,
    MAX(product_length_cm) AS max_length_cm,

    MIN(product_height_cm) AS min_height_cm,
    MAX(product_height_cm) AS max_height_cm,

    MIN(product_width_cm) AS min_width_cm,
    MAX(product_width_cm) AS max_width_cm,

    -- Nulos
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_id,
    SUM(CASE WHEN product_category_name IS NULL THEN 1 ELSE 0 END) AS null_category,
    SUM(CASE WHEN product_name_lenght IS NULL THEN 1 ELSE 0 END) AS null_name_lenght,
    SUM(CASE WHEN product_description_lenght IS NULL THEN 1 ELSE 0 END) AS null_description_lenght,
    SUM(CASE WHEN product_photos_qty IS NULL THEN 1 ELSE 0 END) AS null_photos_qty,
    SUM(CASE WHEN product_weight_g IS NULL THEN 1 ELSE 0 END) AS null_weight_g,
    SUM(CASE WHEN product_length_cm IS NULL THEN 1 ELSE 0 END) AS null_length_cm,
    SUM(CASE WHEN product_height_cm IS NULL THEN 1 ELSE 0 END) AS null_height_cm,
    SUM(CASE WHEN product_width_cm IS NULL THEN 1 ELSE 0 END) AS null_width_cm

FROM bronze.products;

qtd_registros,qtd_product_id,qtd_categorias,min_name_lenght,max_name_lenght,min_description_lenght,max_description_lenght,min_photos_qty,max_photos_qty,min_weight_g,max_weight_g,min_length_cm,max_length_cm,min_height_cm,max_height_cm,min_width_cm,max_width_cm,null_product_id,null_category,null_name_lenght,null_description_lenght,null_photos_qty,null_weight_g,null_length_cm,null_height_cm,null_width_cm
32951,32951,73,5.0,76.0,4.0,3992.0,1.0,20.0,0.0,40425.0,7.0,105.0,2.0,105.0,6.0,118.0,0,610,610,610,610,2,2,2,2


In [0]:
%sql
-- query para analisar dados categoricos da base products
SELECT
    product_category_name,
    COUNT(*) AS qtd_produtos
FROM bronze.products
GROUP BY product_category_name
ORDER BY qtd_produtos DESC;

product_category_name,qtd_produtos
cama_mesa_banho,3029
esporte_lazer,2867
moveis_decoracao,2657
beleza_saude,2444
utilidades_domesticas,2335
automotivo,1900
informatica_acessorios,1639
brinquedos,1411
relogios_presentes,1329
telefonia,1134


In [0]:
%sql
-- query para analisar granularidade da base products
SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT product_id) AS qtd_product_id,
    COUNT(*) - COUNT(DISTINCT product_id) AS qtd_duplicidades
FROM bronze.products;

qtd_registros,qtd_product_id,qtd_duplicidades
32951,32951,0


In [0]:
%sql
-- Incluindo descrição da base products no meu Catalog

COMMENT ON TABLE bronze.products IS
'Dados brutos dos produtos do dataset Brazilian E-Commerce Public Dataset by Olist. Possui uma linha por produto, identificado unicamente por product_id. Contém categoria, características descritivas, quantidade de fotos, peso e dimensões físicas. Origem: olist_products_dataset.csv.';

In [0]:
%sql
-- Incluindo descrição dos campos, com dominio e identificadores de nullos da base products no meu Catalog

ALTER TABLE bronze.products ALTER COLUMN product_id
COMMENT 'Identificador único do produto. Domínio: identificador alfanumérico único e não nulo. Foram observados 32.951 produtos distintos.';

ALTER TABLE bronze.products ALTER COLUMN product_category_name
COMMENT 'Categoria do produto em português. Domínio observado: 73 categorias distintas. Pode apresentar valores nulos.';

ALTER TABLE bronze.products ALTER COLUMN product_name_lenght
COMMENT 'Quantidade de caracteres do nome do produto. Domínio observado: 5 a 76. Pode apresentar valores nulos.';

ALTER TABLE bronze.products ALTER COLUMN product_description_lenght
COMMENT 'Quantidade de caracteres da descrição do produto. Domínio observado: 4 a 3.992. Pode apresentar valores nulos.';

ALTER TABLE bronze.products ALTER COLUMN product_photos_qty
COMMENT 'Quantidade de fotos associadas ao produto. Domínio observado: 1 a 20. Pode apresentar valores nulos.';

ALTER TABLE bronze.products ALTER COLUMN product_weight_g
COMMENT 'Peso do produto em gramas. Domínio observado: 0 a 40.425 gramas. Pode apresentar valores nulos.';

ALTER TABLE bronze.products ALTER COLUMN product_length_cm
COMMENT 'Comprimento do produto em centímetros. Domínio observado: 7 a 105 cm. Pode apresentar valores nulos.';

ALTER TABLE bronze.products ALTER COLUMN product_height_cm
COMMENT 'Altura do produto em centímetros. Domínio observado: 2 a 105 cm. Pode apresentar valores nulos.';

ALTER TABLE bronze.products ALTER COLUMN product_width_cm
COMMENT 'Largura do produto em centímetros. Domínio observado: 6 a 118 cm. Pode apresentar valores nulos.';

#### 03_4_customers

In [0]:
%sql
-- query para buscar dominios da base customers
SELECT
    COUNT(*) AS qtd_registros,

    -- Identificadores
    COUNT(DISTINCT customer_id) AS qtd_customer_id,
    COUNT(DISTINCT customer_unique_id) AS qtd_customer_unique_id,

    -- Localização
    COUNT(DISTINCT customer_zip_code_prefix) AS qtd_ceps,
    COUNT(DISTINCT customer_city) AS qtd_cidades,
    COUNT(DISTINCT customer_state) AS qtd_estados,

    MIN(customer_zip_code_prefix) AS min_zip_code_prefix,
    MAX(customer_zip_code_prefix) AS max_zip_code_prefix,

    -- Nulos
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer_id,
    SUM(CASE WHEN customer_unique_id IS NULL THEN 1 ELSE 0 END) AS null_customer_unique_id,
    SUM(CASE WHEN customer_zip_code_prefix IS NULL THEN 1 ELSE 0 END) AS null_zip_code_prefix,
    SUM(CASE WHEN customer_city IS NULL THEN 1 ELSE 0 END) AS null_city,
    SUM(CASE WHEN customer_state IS NULL THEN 1 ELSE 0 END) AS null_state

FROM bronze.customers;

qtd_registros,qtd_customer_id,qtd_customer_unique_id,qtd_ceps,qtd_cidades,qtd_estados,min_zip_code_prefix,max_zip_code_prefix,null_customer_id,null_customer_unique_id,null_zip_code_prefix,null_city,null_state
99441,99441,96096,14994,4119,27,1003,99990,0,0,0,0,0


In [0]:
%sql
-- query para analisar dado categorico da base customers
SELECT
    customer_state,
    COUNT(*) AS qtd_registros
FROM bronze.customers
GROUP BY customer_state
ORDER BY customer_state;

customer_state,qtd_registros
AC,81
AL,413
AM,148
AP,68
BA,3380
CE,1336
DF,2140
ES,2033
GO,2020
MA,747


In [0]:
%sql
-- query para analisar granularidade da base customers
SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT customer_id) AS qtd_customer_id,
    COUNT(*) - COUNT(DISTINCT customer_id) AS qtd_duplicidades
FROM bronze.customers;

qtd_registros,qtd_customer_id,qtd_duplicidades
99441,99441,0


In [0]:
%sql
-- Relação entre customer_unique_id e customer_id

WITH clientes AS (
    SELECT
        customer_unique_id,
        COUNT(DISTINCT customer_id) AS qtd_customer_id
    FROM bronze.customers
    GROUP BY customer_unique_id
)

SELECT
    qtd_customer_id,
    COUNT(*) AS qtd_customer_unique_id
FROM clientes
GROUP BY qtd_customer_id
ORDER BY qtd_customer_id;

qtd_customer_id,qtd_customer_unique_id
1,93099
2,2745
3,203
4,30
5,8
6,6
7,3
9,1
17,1


In [0]:
%sql
-- Incluindo descrição da base customers no meu Catalog
COMMENT ON TABLE bronze.customers IS
'Dados brutos dos clientes do dataset Brazilian E-Commerce Public Dataset by Olist. Possui uma linha por customer_id e contém o identificador único do consumidor e informações geográficas associadas ao cadastro do cliente no pedido. Origem: olist_customers_dataset.csv.';

In [0]:
%sql
-- Incluindo descrição dos campos, com dominio e identificadores de nullos da base customers no meu Catalog
ALTER TABLE bronze.customers ALTER COLUMN customer_id
COMMENT 'Identificador do cliente associado ao pedido. Domínio: identificador alfanumérico único e não nulo. Foram observados 99.441 customer_id distintos.';

ALTER TABLE bronze.customers ALTER COLUMN customer_unique_id
COMMENT 'Identificador que permite reconhecer o mesmo consumidor em diferentes registros e pedidos. Domínio: identificador alfanumérico não nulo. Foram observados 96.096 customer_unique_id distintos.';

ALTER TABLE bronze.customers ALTER COLUMN customer_zip_code_prefix
COMMENT 'Prefixo do CEP associado ao endereço do cliente no pedido. Domínio observado: valores de 1003 a 99990, com 14.994 valores distintos e sem valores nulos.';

ALTER TABLE bronze.customers ALTER COLUMN customer_city
COMMENT 'Cidade associada ao endereço do cliente no pedido. Domínio observado: 4.119 cidades distintas, sem valores nulos.';

ALTER TABLE bronze.customers ALTER COLUMN customer_state
COMMENT 'Unidade federativa associada ao endereço do cliente no pedido. Domínio observado: 27 estados brasileiros, sem valores nulos.';

#### 03_5_order_payments

In [0]:
%sql
-- Query para entender domininos da base order_payments

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT order_id) AS qtd_order_id,

    -- payment_sequential
    MIN(payment_sequential) AS min_payment_sequential,
    MAX(payment_sequential) AS max_payment_sequential,

    -- payment_installments
    MIN(payment_installments) AS min_payment_installments,
    MAX(payment_installments) AS max_payment_installments,

    -- payment_value
    MIN(payment_value) AS min_payment_value,
    MAX(payment_value) AS max_payment_value,

    -- Nulos
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN payment_sequential IS NULL THEN 1 ELSE 0 END) AS null_payment_sequential,
    SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS null_payment_type,
    SUM(CASE WHEN payment_installments IS NULL THEN 1 ELSE 0 END) AS null_payment_installments,
    SUM(CASE WHEN payment_value IS NULL THEN 1 ELSE 0 END) AS null_payment_value

FROM bronze.order_payments;

qtd_registros,qtd_order_id,min_payment_sequential,max_payment_sequential,min_payment_installments,max_payment_installments,min_payment_value,max_payment_value,null_order_id,null_payment_sequential,null_payment_type,null_payment_installments,null_payment_value
103886,99440,1,29,0,24,0.0,13664.08,0,0,0,0,0


In [0]:
%sql

-- query para analisar dado categorico da base orders_payments

SELECT 
    payment_type,
    COUNT(*) AS qtd_registros
FROM bronze.order_payments
group by 1

payment_type,qtd_registros
credit_card,76795
voucher,5775
boleto,19784
debit_card,1529
not_defined,3


In [0]:
%sql
-- Query para validaçao da granularidade de order_payments

SELECT
    COUNT(*) AS qtd_registros,

    COUNT(
        DISTINCT CONCAT(
            order_id,
            '-',
            CAST(payment_sequential AS STRING)
        )
    ) AS qtd_chaves_pagamento,

    COUNT(*) -
    COUNT(
        DISTINCT CONCAT(
            order_id,
            '-',
            CAST(payment_sequential AS STRING)
        )
    ) AS qtd_duplicidades

FROM bronze.order_payments;

qtd_registros,qtd_chaves_pagamento,qtd_duplicidades
103886,103886,0


In [0]:
%sql
-- Incluindo descrição da base products no meu Catalog
COMMENT ON TABLE bronze.order_payments IS
'Dados brutos dos pagamentos dos pedidos do dataset Brazilian E-Commerce Public Dataset by Olist. Possui uma linha por registro de pagamento de um pedido, sendo a combinação order_id e payment_sequential única na tabela. Um mesmo pedido pode possuir múltiplos registros e diferentes meios de pagamento. Origem: olist_order_payments_dataset.csv.';

In [0]:
%sql
-- Incluindo descrição dos campos, com dominio e identificadores de nullos da base products no meu Catalog
ALTER TABLE bronze.order_payments ALTER COLUMN order_id
COMMENT 'Identificador do pedido ao qual o pagamento está associado. Domínio: identificador alfanumérico não nulo. Foram observados 99.440 pedidos distintos.';

ALTER TABLE bronze.order_payments ALTER COLUMN payment_sequential
COMMENT 'Número sequencial do registro de pagamento dentro do pedido. Domínio observado: valores inteiros de 1 a 29, sem valores nulos. Em conjunto com order_id identifica unicamente um registro de pagamento.';

ALTER TABLE bronze.order_payments ALTER COLUMN payment_type
COMMENT 'Meio de pagamento utilizado. Domínio observado: credit_card, boleto, voucher, debit_card e not_defined. Sem valores nulos.';

ALTER TABLE bronze.order_payments ALTER COLUMN payment_installments
COMMENT 'Quantidade de parcelas associada ao registro de pagamento. Domínio observado: valores de 0 a 24, sem valores nulos.';

ALTER TABLE bronze.order_payments ALTER COLUMN payment_value
COMMENT 'Valor associado ao registro de pagamento. Domínio observado: 0 a 13.664,08, sem valores nulos.';

#### 03_6_order_reviews

In [0]:
%sql
-- Query para catalogação da base order_reviews

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT review_id) AS qtd_review_id,
    COUNT(DISTINCT order_id) AS qtd_order_id,

    -- Score
    MIN(review_score) AS min_review_score,
    MAX(review_score) AS max_review_score,

    -- Datas
    MIN(review_creation_date) AS min_review_creation_date,
    MAX(review_creation_date) AS max_review_creation_date,

    MIN(review_answer_timestamp) AS min_review_answer_timestamp,
    MAX(review_answer_timestamp) AS max_review_answer_timestamp,

    -- Nulos
    SUM(CASE WHEN review_id IS NULL THEN 1 ELSE 0 END) AS null_review_id,
    SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS null_order_id,
    SUM(CASE WHEN review_score IS NULL THEN 1 ELSE 0 END) AS null_review_score,
    SUM(CASE WHEN review_comment_title IS NULL THEN 1 ELSE 0 END) AS null_comment_title,
    SUM(CASE WHEN review_comment_message IS NULL THEN 1 ELSE 0 END) AS null_comment_message,
    SUM(CASE WHEN review_creation_date IS NULL THEN 1 ELSE 0 END) AS null_creation_date,
    SUM(CASE WHEN review_answer_timestamp IS NULL THEN 1 ELSE 0 END) AS null_answer_timestamp

FROM bronze.order_reviews;

qtd_registros,qtd_review_id,qtd_order_id,min_review_score,max_review_score,min_review_creation_date,max_review_creation_date,min_review_answer_timestamp,max_review_answer_timestamp,null_review_id,null_order_id,null_review_score,null_comment_title,null_comment_message,null_creation_date,null_answer_timestamp
99224,98410,98673,1,5,2016-10-02 00:00:00,2018-08-31 00:00:00,2016-10-07 18:32:28,2018-10-29 12:27:35,0,0,0,87656,58247,0,0


In [0]:
%sql
-- Análise da granularidade de order_reviews

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT review_id) AS qtd_review_id,
    COUNT(DISTINCT order_id) AS qtd_order_id,

    COUNT(*) - COUNT(DISTINCT review_id) AS duplicidades_review_id

FROM bronze.order_reviews;

qtd_registros,qtd_review_id,qtd_order_id,duplicidades_review_id
99224,98410,98673,814


In [0]:
%sql
-- Validando a combinação order_id + review_id

SELECT
    COUNT(*) AS qtd_registros,

    COUNT(
        DISTINCT CONCAT(
            order_id,
            '-',
            review_id
        )
    ) AS qtd_chaves_order_review,

    COUNT(*) -
    COUNT(
        DISTINCT CONCAT(
            order_id,
            '-',
            review_id
        )
    ) AS qtd_duplicidades

FROM bronze.order_reviews;

qtd_registros,qtd_chaves_order_review,qtd_duplicidades
99224,99224,0


In [0]:
%sql
-- Incluindo descrição da base order_reviews no meu Catalog
COMMENT ON TABLE bronze.order_reviews IS
'Dados brutos das avaliações dos pedidos do dataset Brazilian E-Commerce Public Dataset by Olist. Possui uma linha por registro de avaliação, sendo a combinação order_id e review_id única na tabela. Um pedido pode possuir mais de uma avaliação. Origem: olist_order_reviews_dataset.csv.';

In [0]:
%sql
-- Incluindo descrição dos campos, com dominio e identificadores de nullos da base order_reviews no meu Catalog
ALTER TABLE bronze.order_reviews ALTER COLUMN review_id
COMMENT 'Identificador da avaliação. Domínio: identificador alfanumérico não nulo. Foram observados 98.410 review_id distintos. O campo isoladamente não identifica unicamente um registro da tabela.';

ALTER TABLE bronze.order_reviews ALTER COLUMN order_id
COMMENT 'Identificador do pedido ao qual a avaliação está associada. Domínio: identificador alfanumérico não nulo. Foram observados 98.673 pedidos distintos.';

ALTER TABLE bronze.order_reviews ALTER COLUMN review_score
COMMENT 'Nota atribuída na avaliação do pedido. Domínio observado: valores inteiros de 1 a 5, sem valores nulos.';

ALTER TABLE bronze.order_reviews ALTER COLUMN review_comment_title
COMMENT 'Título do comentário informado na avaliação. Domínio: texto livre e opcional. Pode apresentar valores nulos.';

ALTER TABLE bronze.order_reviews ALTER COLUMN review_comment_message
COMMENT 'Mensagem textual informada na avaliação. Domínio: texto livre e opcional. Pode apresentar valores nulos.';

ALTER TABLE bronze.order_reviews ALTER COLUMN review_creation_date
COMMENT 'Data de criação da avaliação. Domínio observado: 2016-10-02 00:00:00 a 2018-08-31 00:00:00, sem valores nulos. Na camada Bronze é mantido como STRING.';

ALTER TABLE bronze.order_reviews ALTER COLUMN review_answer_timestamp
COMMENT 'Data e hora de resposta da avaliação. Domínio observado: 2016-10-07 18:32:28 a 2018-10-29 12:27:35, sem valores nulos. Na camada Bronze é mantido como STRING.';

#### 03_7_sellers

In [0]:
%sql
-- Query para catalogação da bronze.sellers

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT seller_id) AS qtd_seller_id,

    COUNT(DISTINCT seller_zip_code_prefix) AS qtd_ceps,
    COUNT(DISTINCT seller_city) AS qtd_cidades,
    COUNT(DISTINCT seller_state) AS qtd_estados,

    MIN(seller_zip_code_prefix) AS min_zip_code_prefix,
    MAX(seller_zip_code_prefix) AS max_zip_code_prefix,

    -- Nulos
    SUM(CASE WHEN seller_id IS NULL THEN 1 ELSE 0 END) AS null_seller_id,
    SUM(CASE WHEN seller_zip_code_prefix IS NULL THEN 1 ELSE 0 END) AS null_zip_code_prefix,
    SUM(CASE WHEN seller_city IS NULL THEN 1 ELSE 0 END) AS null_city,
    SUM(CASE WHEN seller_state IS NULL THEN 1 ELSE 0 END) AS null_state

FROM bronze.sellers;

qtd_registros,qtd_seller_id,qtd_ceps,qtd_cidades,qtd_estados,min_zip_code_prefix,max_zip_code_prefix,null_seller_id,null_zip_code_prefix,null_city,null_state
3095,3095,2246,611,23,1001,99730,0,0,0,0


In [0]:
%sql
-- Validação da granularidade de bronze.sellers

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT seller_id) AS qtd_seller_id,
    COUNT(*) - COUNT(DISTINCT seller_id) AS qtd_duplicidades
FROM bronze.sellers;

qtd_registros,qtd_seller_id,qtd_duplicidades
3095,3095,0


In [0]:
%sql
-- Query para analise de dados categoricos da base sellers
SELECT
    seller_state,
    COUNT(*) AS qtd_vendedores
FROM bronze.sellers
GROUP BY seller_state
ORDER BY seller_state;

seller_state,qtd_vendedores
AC,1
AM,1
BA,19
CE,13
DF,30
ES,23
GO,40
MA,1
MG,244
MS,5


In [0]:
%sql
-- Incluindo descrição da base sellers no meu Catalog
COMMENT ON TABLE bronze.sellers IS
'Dados brutos dos vendedores do dataset Brazilian E-Commerce Public Dataset by Olist. Possui uma linha por vendedor, identificado unicamente por seller_id, e contém informações geográficas associadas ao vendedor. Origem: olist_sellers_dataset.csv.';

In [0]:
%sql
-- Incluindo descrição dos campos, com dominio e identificadores de nullos da base sellers no meu Catalog
ALTER TABLE bronze.sellers ALTER COLUMN seller_id
COMMENT 'Identificador único do vendedor. Domínio: identificador alfanumérico único e não nulo. Foram observados 3.095 vendedores distintos.';

ALTER TABLE bronze.sellers ALTER COLUMN seller_zip_code_prefix
COMMENT 'Prefixo do CEP associado ao vendedor. Domínio observado: valores de 1001 a 99730, com 2.246 valores distintos e sem valores nulos.';

ALTER TABLE bronze.sellers ALTER COLUMN seller_city
COMMENT 'Cidade associada ao vendedor. Domínio observado: 611 cidades distintas, sem valores nulos.';

ALTER TABLE bronze.sellers ALTER COLUMN seller_state
COMMENT 'Unidade federativa associada ao vendedor. Domínio observado: 23 estados brasileiros, sem valores nulos.';

#### 03_8_categoy_translation

In [0]:
%sql
-- Query para catalogação da bronze.category_translation

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT product_category_name) AS qtd_categorias_pt,
    COUNT(DISTINCT product_category_name_english) AS qtd_categorias_en,

    SUM(CASE WHEN product_category_name IS NULL THEN 1 ELSE 0 END) AS null_categoria_pt,
    SUM(CASE WHEN product_category_name_english IS NULL THEN 1 ELSE 0 END) AS null_categoria_en

FROM bronze.category_translation;

qtd_registros,qtd_categorias_pt,qtd_categorias_en,null_categoria_pt,null_categoria_en
71,71,71,0,0


In [0]:
%sql
-- Validação da granularidade

SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT product_category_name) AS qtd_categorias_pt,
    COUNT(*) - COUNT(DISTINCT product_category_name) AS qtd_duplicidades

FROM bronze.category_translation;

qtd_registros,qtd_categorias_pt,qtd_duplicidades
71,71,0


In [0]:
%sql
-- Categorias de produtos sem correspondência na tabela de tradução

SELECT
    p.product_category_name,
    COUNT(*) AS qtd_produtos
FROM bronze.products p
LEFT JOIN bronze.category_translation t
    ON p.product_category_name = t.product_category_name
WHERE p.product_category_name IS NOT NULL
  AND t.product_category_name IS NULL
GROUP BY p.product_category_name
ORDER BY qtd_produtos DESC;

product_category_name,qtd_produtos
portateis_cozinha_e_preparadores_de_alimentos,10
pc_gamer,3


In [0]:
%sql
-- Incluindo descrição da base category_translation no meu Catalog
COMMENT ON TABLE bronze.category_translation IS
'Tabela de referência para tradução das categorias de produtos do português para o inglês, pertencente ao dataset Brazilian E-Commerce Public Dataset by Olist. Possui uma linha por categoria em português. Origem: product_category_name_translation.csv.';

In [0]:
%sql
-- Incluindo descrição dos campos, com dominio e identificadores de nullos da base category_translation no meu Catalog
ALTER TABLE bronze.category_translation ALTER COLUMN product_category_name
COMMENT 'Nome da categoria do produto em português. Domínio observado: 71 categorias distintas, sem valores nulos e sem duplicidades.';

ALTER TABLE bronze.category_translation ALTER COLUMN product_category_name_english
COMMENT 'Nome da categoria do produto traduzido para o inglês. Domínio observado: 71 categorias distintas, sem valores nulos.';

### 04_discovery_datasets
Nesta etapa são realizados análises exploratórias adicionais sobre a estrutura e o relacionamento entre os datasets. O objetivo é compreender diferenças de granularidade, cardinalidade e situações relevantes para a definição do modelo analítico e das regras de transformação. As análises realizadas nesta etapa subsidiaram as decisões de modelagem e qualidade aplicadas nos notebooks posteriores, não representando ainda o tratamento dos dados.

In [0]:
%sql
-- Analise para contagem de linhas (entender granularidade de cada dataset para estabeler meu desenho de star schema)
SELECT 'orders' AS tabela, COUNT(*) AS linhas FROM bronze.orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM bronze.order_items
UNION ALL
SELECT 'products', COUNT(*) FROM bronze.products
UNION ALL
SELECT 'customers', COUNT(*) FROM bronze.customers
UNION ALL
SELECT 'sellers', COUNT(*) FROM bronze.sellers
UNION ALL
SELECT 'order_reviews', COUNT(*) FROM bronze.order_reviews
UNION ALL
SELECT 'order_payments', COUNT(*) FROM bronze.order_payments
UNION ALL
SELECT 'category_translation', COUNT(*) FROM bronze.category_translation;

tabela,linhas
orders,99441
order_items,112650
products,32951
customers,99441
sellers,3095
order_reviews,99224
order_payments,103886
category_translation,71


In [0]:
%sql
-- Analise para contagem de linhas (entender granularidade de cada dataset para estabeler meu desenho de star schema)
SELECT
    'orders' AS tabela,
    COUNT(*) AS linhas,
    COUNT(DISTINCT order_id) AS pedidos
FROM bronze.orders

UNION ALL

SELECT
    'order_items',
    COUNT(*),
    COUNT(DISTINCT order_id)
FROM bronze.order_items

UNION ALL

SELECT
    'order_reviews',
    COUNT(*),
    COUNT(DISTINCT order_id)
FROM bronze.order_reviews

UNION ALL

SELECT
    'order_payments',
    COUNT(*),
    COUNT(DISTINCT order_id)
FROM bronze.order_payments;

tabela,linhas,pedidos
orders,99441,99441
order_items,112650,98666
order_reviews,99224,98673
order_payments,103886,99440


In [0]:
%sql
-- Analise para contagem de linhas (entender granularidade de cada dataset para estabeler meu desenho de star schema)

-- Quantos pedidos possuem múltiplos itens?
SELECT
    'Mais de 1 item' AS analise,
    COUNT(*) AS qtd_pedidos
FROM (
    SELECT order_id
    FROM bronze.order_items
    GROUP BY order_id
    HAVING COUNT(*) > 1
)

UNION ALL

-- Quantos pedidos possuem itens de mais de um seller?
SELECT
    'Mais de 1 seller',
    COUNT(*)
FROM (
    SELECT order_id
    FROM bronze.order_items
    GROUP BY order_id
    HAVING COUNT(DISTINCT seller_id) > 1
)

UNION ALL

-- Quantos pedidos possuem mais de um registro de pagamento?
SELECT
    'Mais de 1 pagamento',
    COUNT(*)
FROM (
    SELECT order_id
    FROM bronze.order_payments
    GROUP BY order_id
    HAVING COUNT(*) > 1
)

UNION ALL

-- Quantos pedidos utilizaram mais de um TIPO de pagamento?
SELECT
    'Mais de 1 tipo de pagamento',
    COUNT(*)
FROM (
    SELECT order_id
    FROM bronze.order_payments
    GROUP BY order_id
    HAVING COUNT(DISTINCT payment_type) > 1
)

UNION ALL

-- Quantos pedidos possuem mais de uma avaliação?
SELECT
    'Mais de 1 review',
    COUNT(*)
FROM (
    SELECT order_id
    FROM bronze.order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
);

analise,qtd_pedidos
Mais de 1 item,9803
Mais de 1 seller,1278
Mais de 1 pagamento,2961
Mais de 1 tipo de pagamento,2246
Mais de 1 review,547


In [0]:
%sql
-- Avaliando comportamento da base de reviews para desenha solução de negocio para tratar pedidos com mais de 1 review
WITH reviews_por_pedido AS (
    SELECT
        order_id,
        COUNT(*) AS qtd_reviews,
        COUNT(DISTINCT review_score) AS qtd_scores_distintos,
        MIN(review_score) AS menor_score,
        MAX(review_score) AS maior_score
    FROM bronze.order_reviews
    GROUP BY order_id
)

SELECT
    qtd_reviews,
    COUNT(*) AS qtd_pedidos,
    SUM(
        CASE 
            WHEN qtd_scores_distintos > 1 THEN 1 
            ELSE 0 
        END
    ) AS pedidos_com_scores_diferentes
FROM reviews_por_pedido
GROUP BY qtd_reviews
ORDER BY qtd_reviews;

qtd_reviews,qtd_pedidos,pedidos_com_scores_diferentes
1,98126,0
2,543,200
3,4,2


In [0]:
%sql
-- Avaliando comportamento da base de reviews para desenha solução de negocio para tratar pedidos com mais de 1 review
-- Aqui trazendo a media porque estou considerando aplicar a regra de negocio de aplicar a media dos scores
WITH base_review as (
SELECT
    order_id,
    COUNT(*) AS qtd_reviews,
    COUNT(DISTINCT review_score) AS qtd_scores_distintos,
    MIN(review_score) AS menor_score,
    MAX(review_score) AS maior_score
FROM bronze.order_reviews
GROUP BY order_id
HAVING COUNT(*) > 1
)
,med_reviews as (
SELECT
    order_id,
    avg(review_score) as med_review_score
FROM bronze.order_reviews
GROUP BY 1
)

SELECT
    a.order_id,
    a.qtd_reviews,
    a.qtd_scores_distintos,
    a.menor_score,
    a.maior_score,
    b.med_review_score
FROM base_review a
LEFT JOIN med_reviews b
    ON a.order_id = b.order_id
ORDER BY a.qtd_reviews DESC, a.qtd_scores_distintos DESC



order_id,qtd_reviews,qtd_scores_distintos,menor_score,maior_score,med_review_score
c88b1d1b157a9999ce368f218a407141,3,2,3,5,4.333333333333333
03c939fd7fd3b38f8485a0f95798f1f6,3,2,3,4,3.3333333333333335
8e17072ec97ce29f0e1f111e598b0c85,3,1,1,1,1.0
df56136b8031ecd28e200bb18e6ddb2e,3,1,5,5,5.0
2ce757e7e2cf3442241e4646b3472dd0,2,2,4,5,4.5
7139a3b215ea730e2db624cc99820578,2,2,2,4,3.0
c761a8b74f1e876bc5efc4186f720e27,2,2,1,4,2.5
ca263afd88a8a1200605adbd4b63cd7d,2,2,1,4,2.5
9a19e859f5bd7a0bb22d64d35fa8b979,2,2,1,4,2.5
ce0102221c8d12a97979c74d09cca282,2,2,2,3,2.5


In [0]:
%sql
-- Avaliando comportamento da base de pagamentos | Tipos de pagamento e o peso que cada um representa
SELECT
    payment_type,
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT order_id) AS qtd_pedidos,
    ROUND(SUM(payment_value), 2) AS valor_pagamentos,
    ROUND(AVG(payment_value), 2) AS valor_medio_pagamento
FROM bronze.order_payments
GROUP BY payment_type
ORDER BY qtd_pedidos DESC;

payment_type,qtd_registros,qtd_pedidos,valor_pagamentos,valor_medio_pagamento
credit_card,76795,76505,1.254208419E7,163.32
boleto,19784,19784,2869361.27,145.03
voucher,5775,3866,379436.87,65.7
debit_card,1529,1528,217989.79,142.57
not_defined,3,3,0.0,0.0


In [0]:
%sql
-- Avaliando comportamento da base de pagamentos | Parcelamento por tipo de pagamento
SELECT
    payment_type,
    payment_installments,
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT order_id) AS qtd_pedidos,
    ROUND(AVG(payment_value), 2) AS valor_medio
FROM bronze.order_payments
GROUP BY
    payment_type,
    payment_installments
ORDER BY
    payment_type,
    payment_installments;

payment_type,payment_installments,qtd_registros,qtd_pedidos,valor_medio
boleto,1,19784,19784,145.03
credit_card,0,2,2,94.32
credit_card,1,25455,25407,95.87
credit_card,2,12413,12389,127.23
credit_card,3,10461,10443,142.54
credit_card,4,7098,7088,163.98
credit_card,5,5239,5234,183.47
credit_card,6,3920,3916,209.85
credit_card,7,1626,1623,187.67
credit_card,8,4268,4253,307.74


In [0]:
%sql
-- Avaliando comportamento da base de pagamentos | Entender quantidade de tipos de pagamento por pedido
WITH pagamentos_pedido AS (
    SELECT
        order_id,
        COUNT(*) AS qtd_pagamentos,
        COUNT(DISTINCT payment_type) AS qtd_tipos_pagamento,
        SUM(payment_value) AS valor_total_pagamento,
        MAX(payment_installments) AS max_parcelas
    FROM bronze.order_payments
    GROUP BY order_id
)

SELECT
    qtd_pagamentos,
    qtd_tipos_pagamento,
    COUNT(*) AS qtd_pedidos,
    ROUND(AVG(valor_total_pagamento), 2) AS valor_medio_pedido
FROM pagamentos_pedido
GROUP BY
    qtd_pagamentos,
    qtd_tipos_pagamento
ORDER BY
    qtd_pagamentos,
    qtd_tipos_pagamento;


qtd_pagamentos,qtd_tipos_pagamento,qtd_pedidos,valor_medio_pedido
1,1,96479,160.83
2,1,512,254.65
2,2,1870,151.3
3,1,86,89.52
3,2,215,141.05
4,1,46,92.69
4,2,62,111.45
5,1,19,91.87
5,2,33,150.49
6,1,15,176.63


In [0]:
%sql
-- Avaliando comportamento da base de pagamentos | Comparando payment_value com (price + freight_value) da order_items
WITH itens AS (
    SELECT
        order_id,
        SUM(price) AS valor_produtos,
        SUM(freight_value) AS valor_frete,
        SUM(price + freight_value) AS valor_total_itens
    FROM bronze.order_items
    GROUP BY order_id
),

pagamentos AS (
    SELECT
        order_id,
        SUM(payment_value) AS valor_total_pagamento
    FROM bronze.order_payments
    GROUP BY order_id
)

SELECT
    COUNT(*) AS qtd_pedidos_comparados,

    SUM(
        CASE
            WHEN ABS(i.valor_total_itens - p.valor_total_pagamento) < 0.01
            THEN 1 ELSE 0
        END
    ) AS valores_iguais,

    SUM(
        CASE
            WHEN ABS(i.valor_total_itens - p.valor_total_pagamento) >= 0.01
            THEN 1 ELSE 0
        END
    ) AS valores_diferentes,

    ROUND(AVG(
        ABS(i.valor_total_itens - p.valor_total_pagamento)
    ), 2) AS diferenca_media,

    ROUND(MAX(
        ABS(i.valor_total_itens - p.valor_total_pagamento)
    ), 2) AS maior_diferenca

FROM itens i
INNER JOIN pagamentos p
    ON i.order_id = p.order_id;

qtd_pedidos_comparados,valores_iguais,valores_diferentes,diferenca_media,maior_diferenca
98665,98285,380,0.03,182.81


In [0]:
%sql
-- Avaliando comportamento da base de pagamentos | Comparando payment_value com (price + freight_value) da order_items
WITH itens AS (
    SELECT
        order_id,
        SUM(price) AS valor_produtos,
        SUM(freight_value) AS valor_frete,
        SUM(price + freight_value) AS valor_total_itens
    FROM bronze.order_items
    GROUP BY order_id
),

pagamentos AS (
    SELECT
        order_id,
        SUM(payment_value) AS valor_total_pagamento,
        COUNT(*) AS qtd_pagamentos,
        COUNT(DISTINCT payment_type) AS qtd_tipos_pagamento
    FROM bronze.order_payments
    GROUP BY order_id
)

SELECT
    i.order_id,
    ROUND(i.valor_produtos, 2) AS valor_produtos,
    ROUND(i.valor_frete, 2) AS valor_frete,
    ROUND(i.valor_total_itens, 2) AS valor_total_itens,
    ROUND(p.valor_total_pagamento, 2) AS valor_total_pagamento,
    ROUND(p.valor_total_pagamento - i.valor_total_itens, 2) AS diferenca,
    p.qtd_pagamentos,
    p.qtd_tipos_pagamento
FROM itens i
INNER JOIN pagamentos p
    ON i.order_id = p.order_id
WHERE ABS(i.valor_total_itens - p.valor_total_pagamento) >= 0.01
ORDER BY ABS(p.valor_total_pagamento - i.valor_total_itens) DESC
LIMIT 30;

order_id,valor_produtos,valor_frete,valor_total_itens,valor_total_pagamento,diferenca,qtd_pagamentos,qtd_tipos_pagamento
ce6d150fb29ada17d2082f4847107665,1299.0,104.66,1403.66,1586.47,182.81,1,1
6e5fe7366a2e1bfbf3257dba0af1267f,179.19,108.72,287.91,406.92,119.01,1,1
70b742795bc441e94a44a084b6d9ce7a,269.99,196.94,466.93,578.82,111.89,1,1
996c7e73600ad3723e8627ab7bef81e4,559.9,28.0,587.9,664.43,76.53,1,1
70b7e94ea46d3e8b5bc12a50186edaf0,167.88,45.27,213.15,274.84,61.69,1,1
bc2c82b0ef78d2252b6176d1972db7c9,165.0,77.01,242.01,303.02,61.01,1,1
af9ffff2ce6b3defd34fd4c78857a379,395.65,17.52,413.17,466.97,53.8,1,1
262118ce178bb3e4590a3adcf6d62e6b,119.8,57.94,177.74,126.12,-51.62,1,1
bfdb5bbb06458d600a33d61f5f287472,297.0,51.93,348.93,394.36,45.43,1,1
8d9c0dc8d5a2ce804f6b925d8f8e6c1d,209.8,44.65,254.45,293.89,39.44,1,1


In [0]:
%sql
-- Avaliando comportamento da base customer para definir chave unica
SELECT
    COUNT(*) AS qtd_registros,
    COUNT(DISTINCT customer_id) AS qtd_customer_id,
    COUNT(DISTINCT customer_unique_id) AS qtd_customer_unique_id
FROM bronze.customers;

qtd_registros,qtd_customer_id,qtd_customer_unique_id
99441,99441,96096


In [0]:
%sql
-- Avaliando recompra com os campos de customers
WITH pedidos_cliente AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS qtd_pedidos
    FROM bronze.orders o
    INNER JOIN bronze.customers c
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_unique_id
)

SELECT
    qtd_pedidos,
    COUNT(*) AS qtd_clientes
FROM pedidos_cliente
GROUP BY qtd_pedidos
ORDER BY qtd_pedidos;

qtd_pedidos,qtd_clientes
1,93099
2,2745
3,203
4,30
5,8
6,6
7,3
9,1
17,1


In [0]:
%sql
-- Avaliando mudanças geograficas para o mesmo customer_unique_id
SELECT
    customer_unique_id,
    COUNT(DISTINCT customer_id) AS qtd_customer_ids,
    COUNT(DISTINCT customer_city) AS qtd_cidades,
    COUNT(DISTINCT customer_state) AS qtd_estados
FROM bronze.customers
GROUP BY customer_unique_id
HAVING
       COUNT(DISTINCT customer_id) > 1
    OR COUNT(DISTINCT customer_city) > 1
    OR COUNT(DISTINCT customer_state) > 1
ORDER BY qtd_customer_ids DESC, qtd_estados DESC, qtd_cidades DESC
LIMIT 50;

customer_unique_id,qtd_customer_ids,qtd_cidades,qtd_estados
8d50f5eadf50201ccdcedfb9e2ac8455,17,1,1
3e43e6105506432c953e165fb2acf44c,9,1,1
ca77025e7201e3b30c44b472ff346268,7,1,1
1b6c7548a2a1f9037c1fd3ddfed95f33,7,1,1
6469f99c1f9dfae7733b25662e7f1782,7,1,1
63cfc61cee11cbe306bff5857d00bfe4,6,1,1
47c1a3033b8b77b3ab6e109eb4d5fdf3,6,1,1
dc813062e0fc23409cd255f7f53c7074,6,1,1
de34b16117594161a6a89c50b289d35a,6,1,1
12f5d6e1cbf93dafd9dcc19095df0b3d,6,1,1


In [0]:
%sql

WITH localizacao_cliente AS (
    SELECT
        customer_unique_id,
        COUNT(DISTINCT customer_city) AS qtd_cidades,
        COUNT(DISTINCT customer_state) AS qtd_estados
    FROM bronze.customers
    GROUP BY customer_unique_id
)

SELECT
    COUNT(*) AS total_clientes,
    
    SUM(
        CASE WHEN qtd_cidades > 1
        THEN 1 ELSE 0 END
    ) AS clientes_multiplas_cidades,
    
    SUM(
        CASE WHEN qtd_estados > 1
        THEN 1 ELSE 0 END
    ) AS clientes_multiplos_estados

FROM localizacao_cliente;

total_clientes,clientes_multiplas_cidades,clientes_multiplos_estados
96096,122,39
